# API for Frontend


In [ ]:
# Install the packages needed to run the web server and expose it publicly
!pip install fastapi uvicorn pyngrok nest-asyncio python-multipart langchain-chroma langchain-core

import uvicorn
import nest_asyncio
from fastapi import FastAPI, File, UploadFile, Form
from fastapi.staticfiles import StaticFiles
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
from pyngrok import ngrok, conf
from PIL import Image
import torch
import io
import os

# nest_asyncio lets uvicorn run inside it without crashing
nest_asyncio.apply()

from google.colab import userdata
conf.get_default().auth_token = userdata.get('NGROK_TOKEN')

app = FastAPI()

# Without this the browser would block requests from the frontend to this server
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# Path to the pre-built React app.
# before running this cell — that's how frontend changes make it into the live app.
DIST_PATH = "/content/Mills-Museum/src/frontend/mcam-keyword-generator/dist"

@app.get("/")
async def serve_frontend():
    return FileResponse(f"{DIST_PATH}/index.html")

# Serve the JS/CSS bundles that index.html references
app.mount("/assets", StaticFiles(directory=f"{DIST_PATH}/assets"), name="assets")

@app.post("/predict")
async def predict(file: UploadFile = File(...), term_count: int = Form(default=20)):
    contents = await file.read()
    image = Image.open(io.BytesIO(contents)).convert("RGB")

    art_query = {"image": image, "text": ""}
    query_input = [art_query]

    image_features = embedding_model.process(query_input)
    image_features = torch.nn.functional.normalize(image_features, p=2, dim=1)  # required for cosine similarity
    query_embedding = image_features.cpu().float().squeeze().tolist()

    # MMR retrieves term_count keywords while penalizing redundancy.
    # fetch_k * 4 gives a larger candidate pool; lambda_mult=0.96 keeps
    # results close to the query while filtering near-duplicates.
    mmr_docs = vectorstore.max_marginal_relevance_search_by_vector(
        embedding=query_embedding,
        k=term_count,
        fetch_k=term_count * 4,
        lambda_mult=0.96,
    )

    labels = []
    input_docs = []
    for doc in mmr_docs:
        # Use metadata label if available, otherwise fall back to first line of doc content
        term_label = doc.metadata.get("term_label", doc.page_content.split("\n")[0])
        input_docs.append({"text": doc.page_content})
        labels.append(term_label)

    rerank_inputs = {
        "instruction": "Retrieve Art & Architecture Thesaurus terms relevant to the given image.",
        "query": art_query,
        "documents": input_docs,
        "fps": 1.0,
    }

    scores = reranking_model.process(rerank_inputs)
    sorted_results = sorted(zip(scores, labels), reverse=True)

    # Response shape matches what keywordAdapters.js on the frontend expects
    keywords = [
        {"label": label, "score": round(float(score) * 100, 1)}
        for score, label in sorted_results
    ]

    return {"keywords": keywords}

# Static ngrok domain so the URL stays the same every run —
# the frontend hardcodes this as its API base URL
public_url = ngrok.connect(8000)
print(f"Public URL: {public_url}")
print(f"Open this URL in your browser to use the app!")

# host="0.0.0.0" makes the server reachable from outside the Colab container
config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()

INFO:     Started server process [7084]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Public API URL: NgrokTunnel: "https://stilliform-celine-plaintively.ngrok-free.dev" -> "http://localhost:8000"
INFO:     144.91.51.54:0 - "OPTIONS /predict HTTP/1.1" 200 OK
INFO:     144.91.51.54:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     144.91.51.54:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     144.91.51.54:0 - "OPTIONS /predict HTTP/1.1" 200 OK
INFO:     144.91.51.54:0 - "POST /predict HTTP/1.1" 200 OK
